# Project 06: Delta Lake Fundamentals & Enterprise Lakehouse Architecture

> **Data Engineering Diaries** | Interactive Reference Implementation

Welcome to the Delta Lake hands-on tutorial notebook. This notebook demonstrates production Delta Lake features:
1. **Delta SparkSession Setup**: Configuring extensions and catalog.
2. **ACID Table Operations**: Create, read, append, and overwrite.
3. **Schema Evolution**: Schema enforcement, `mergeSchema`, and `overwriteSchema`.
4. **Time Travel**: `VERSION AS OF`, history tracking, and table restore.
5. **MERGE INTO**: SCD Type 1 upserts and CDC processing.
6. **Performance Tuning**: `OPTIMIZE` and `Z-ORDER BY` clustering.
7. **Maintenance**: `VACUUM` and retention policies.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Initialize Delta SparkSession
builder = (
    SparkSession.builder
    .appName("DeltaLakeNotebookDemo")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print(f"Spark Version: {spark.version}")
print("Delta Lake extensions configured successfully!")

In [ ]:
# 1. Create and Write Delta Table
import tempfile
table_dir = tempfile.mkdtemp(prefix="nb_delta_")
table_path = os.path.join(table_dir, "events_delta")

data_v0 = [
    ("evt-101", "usr-1", "click", 0.0),
    ("evt-102", "usr-2", "purchase", 99.50)
]
df_v0 = spark.createDataFrame(data_v0, ["event_id", "user_id", "event_type", "amount"])

df_v0.write.format("delta").mode("overwrite").save(table_path)
print(f"Written {df_v0.count()} rows to Delta table at: {table_path}")

# Read table back
read_df = spark.read.format("delta").load(table_path)
read_df.show()

In [ ]:
# Append new data to create Version 1
data_v1 = [
    ("evt-103", "usr-3", "signup", 0.0),
    ("evt-104", "usr-1", "purchase", 149.00)
]
df_v1 = spark.createDataFrame(data_v1, ["event_id", "user_id", "event_type", "amount"])
df_v1.write.format("delta").mode("append").save(table_path)

# Query Version 0 using Time Travel
df_v0_read = spark.read.format("delta").option("versionAsOf", 0).load(table_path)
print("--- Snapshot at Version 0 ---")
df_v0_read.show()

# Query latest version (Version 1)
df_v1_read = spark.read.format("delta").load(table_path)
print("--- Latest Snapshot (Version 1) ---")
df_v1_read.show()

# Inspect Transaction History log
from delta.tables import DeltaTable
delta_table = DeltaTable.forPath(spark, table_path)
delta_table.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

In [ ]:
# Upsert using MERGE INTO (Update evt-102 amount to 120.00, insert new evt-105)
updates_data = [
    ("evt-102", "usr-2", "purchase", 120.00),
    ("evt-105", "usr-4", "click", 0.0)
]
df_updates = spark.createDataFrame(updates_data, ["event_id", "user_id", "event_type", "amount"])

(
    delta_table.alias("target")
    .merge(df_updates.alias("source"), "target.event_id = source.event_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("--- After MERGE INTO Upsert ---")
delta_table.toDF().show()

In [ ]:
# Run OPTIMIZE Z-ORDER BY user_id
print("Executing OPTIMIZE Z-ORDER BY user_id...")
delta_table.optimize().executeZOrderBy("user_id")

# Execute VACUUM RETAIN 168 HOURS DRY RUN
print("Executing VACUUM DRY RUN...")
vacuum_preview = spark.sql(f"VACUUM delta.`{table_path}` RETAIN 168 HOURS DRY RUN")
vacuum_preview.show(truncate=False)